# Seattle Weather Time Series Analysis and Forecasting with Explainability
This notebook demonstrates a complete time series analysis and forecasting workflow on Seattle weather data. We’ll preprocess the data, create sequences, build an LSTM model, and interpret the model with Captum’s Integrated Gradients.

## Workflow Overview
1. **Data Loading and Exploration** - Load and understand the dataset.
2. **Preprocessing** - Encode categorical features, scale numerical values, and create sequences for time series modeling.
3. **Modeling** - Train an LSTM model to predict future weather conditions.
4. **Evaluation and Interpretation** - Evaluate model performance and interpret feature contributions with Integrated Gradients.

Let's get started!


## 1. Data Loading and Exploration
First, let's load the dataset and explore it.

In [ ]:
# Import necessary libraries
import pandas as pd

# Load the dataset
file_path = 'seattle-weather.csv'
data = pd.read_csv(file_path)

# Display the first few rows of the dataset
data.head()

In [ ]:
# Display general info and statistics
data.info()
data.describe()

## 2. Data Preprocessing

Preprocessing is an essential step in any machine learning workflow, as it helps ensure that the data is in the best possible form for a model to learn from it. In this section, we will:

1. **Convert the `date` Column to Datetime Format**
   - The `date` column in the dataset is currently in a text format (string). For time series analysis, it’s essential that the `date` column is recognized as a datetime object.
   - Converting this column allows us to perform time-based operations, like resampling or plotting over time. Additionally, it helps align the dataset with models and visualizations that can handle dates accurately.

2. **Encode the `weather` Column**
   - The `weather` column contains categorical data representing different weather types, such as "drizzle," "fog," "rain," etc.
   - Machine learning models generally cannot process categorical labels directly, so we need to convert these into a numerical format. We use **Label Encoding** to assign a unique integer to each category in `weather`.
   - Here’s the mapping between each weather type and its assigned integer:

| Weather Type | Encoded Value |
|--------------|---------------|
| Drizzle      | 0             |
| Fog          | 1             |
| Rain         | 2             |
| Snow         | 3             |
| Sun          | 4             |


   - This encoding allows the model to treat weather types as numerical data, where each integer represents a unique weather category. Note that these values are arbitrary and do not imply any particular order or relationship between categories.



3. **Scale Numerical Features**
   - The dataset includes numerical features such as `precipitation`, `temp_max`, `temp_min`, and `wind`. These features have varying ranges, which could negatively impact the model’s performance. For instance, `temperature` values may be in the range of 0–40, while `precipitation` might range between 0–10.
   - **Min-Max Scaling** helps normalize the values of each feature to fall within the range of 0 and 1. This ensures that each feature contributes equally to the model, preventing any one feature from disproportionately influencing the model due to its scale.

4. **Drop the Original `weather` Column**
   - After encoding the `weather` column, the original categorical data is no longer needed. We drop this column to clean up the dataset and avoid redundancy.


After completing these steps, we have a dataset that is fully numeric and ready for modeling.


In [ ]:
# Convert 'date' to datetime and encode 'weather' column
data['date'] = pd.to_datetime(data['date'])

# Encode categorical 'weather' column
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
label_encoder = LabelEncoder()
data['weather_encoded'] = label_encoder.fit_transform(data['weather'])

# Scale numerical features
scaler = MinMaxScaler()
data[['precipitation', 'temp_max', 'temp_min', 'wind']] = scaler.fit_transform(
    data[['precipitation', 'temp_max', 'temp_min', 'wind']]
)

# Drop the original 'weather' column
data = data.drop(columns=['weather'])
data.head()

## 3. Sequence Creation

For time series data, models like Long Short-Term Memory (LSTM) networks rely on sequences of data as input. This section explains how to create sequences of weather data for training the LSTM model.

### Why Create Sequences?
- LSTMs and other recurrent neural networks (RNNs) use sequential data to learn temporal patterns over time.
- In this case, each sequence represents a block of 30 consecutive days of weather data, enabling the model to capture patterns in weather over roughly a one-month period.

### Steps to Create Sequences

1. **Define the Sequence Creation Function**
   - We define a function called `create_sequences` that takes the data, a sequence length (in days), and a target index (representing the column we want to predict).
   - This function iterates through the dataset and constructs a sequence of length `seq_length` for each possible starting point.
   - Each sequence is stored in a list (`sequences`), and the corresponding label (i.e., the target value) is stored in another list (`labels`).



In [ ]:
import numpy as np

def create_sequences(data, seq_length=30, target_index=1):
    sequences, labels = [], []
    for i in range(len(data) - seq_length):
        sequences.append(data.iloc[i:i + seq_length].values)
        labels.append(data.iloc[i + seq_length, target_index])
    return np.array(sequences), np.array(labels)

# Set sequence length and create sequences
sequence_length = 30
sequences, labels = create_sequences(data[['precipitation', 'temp_max', 'temp_min', 'wind', 'weather_encoded']], sequence_length)

# Split into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(sequences, labels, test_size=0.2, random_state=42)


## 4. Data Visualization

Visualizing data is crucial for understanding trends, distributions, and patterns in the dataset. In this section, we’ll create plots to examine the temperature trends, precipitation distribution, and frequency of different weather types.


In [ ]:
import matplotlib.pyplot as plt

# Temperature trends over time
plt.figure(figsize=(10, 5))
plt.plot(data['date'], data['temp_max'], label='Max Temp')
plt.plot(data['date'], data['temp_min'], label='Min Temp')
plt.title('Temperature Trends Over Time')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.show()

# Precipitation distribution
plt.figure(figsize=(10, 5))
plt.hist(data['precipitation'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of Precipitation')
plt.xlabel('Precipitation (mm)')
plt.ylabel('Frequency')
plt.show()

# Weather type frequency
# Create a mapping of encoded values back to original weather names
weather_mapping = dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))
weather_counts = data['weather_encoded'].value_counts().sort_index()

# Plot with names and frequencies
plt.figure(figsize=(10, 5))
weather_counts.plot(kind='bar', color='lightgreen', edgecolor='black')
plt.title('Frequency of Weather Types')
plt.xlabel('Weather Type')
plt.ylabel('Frequency')

# Set x-ticks to display weather names instead of encoded numbers
plt.xticks(ticks=range(len(weather_mapping)), labels=[weather_mapping[i] for i in range(len(weather_mapping))], rotation=45)

# Annotate bars with frequencies
for i, count in enumerate(weather_counts):
    plt.text(i, count + 5, str(count), ha='center', va='bottom')

plt.show()



## 5. LSTM Model Building and Training

In this section, we define and train an LSTM (Long Short-Term Memory) model for time series forecasting. LSTMs are effective for learning from sequential data, which makes them suitable for tasks involving temporal patterns, like weather forecasting.

### Steps to Build and Train the LSTM Model

1. **Define the LSTM Model Architecture**
   - The LSTM model includes an LSTM layer and a fully connected (FC) layer for output. Here’s the structure:
     - **LSTM Layer**: Learns patterns over time in the sequence data.
     - **Fully Connected Layer**: Outputs a single prediction based on the learned patterns.

2. **Model Hyperparameters**
   - **input_size**: The number of input features (e.g., precipitation, temperature, etc.).
   - **hidden_size**: The number of units in the LSTM’s hidden layers. A higher value allows the model to capture more complexity.
   - **num_layers**: The number of LSTM layers in the model.
   - **output_size**: The number of outputs from the model; in this case, it’s 1 because we are predicting a single value.

3. **Define the Loss Function and Optimizer**
   - **Loss Function**: We use Mean Squared Error (MSE), which is commonly used for regression tasks.
   - **Optimizer**: Adam optimizer is selected for efficient gradient descent.

4. **DataLoader for Batching**
   - A DataLoader is used to manage batching and shuffling of the training data.

5. **Training the Model**
   - In each epoch, the model learns from batches of data by minimizing the loss.

   - During each epoch:
     - The model makes predictions (`outputs`).
     - The **loss** is calculated as the difference between predictions and actual values.
     - The gradients are backpropagated to update model weights, minimizing the loss.



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Define LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        out = self.fc(hn[-1])
        return out

# Model hyperparameters
input_size = X_train.shape[2]
hidden_size = 50
num_layers = 2
output_size = 1
num_epochs = 10
batch_size = 32
learning_rate = 0.001

# Initialize model, criterion, optimizer
model = LSTMModel(input_size, hidden_size, num_layers, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# DataLoader for batching
train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)),
                          batch_size=batch_size, shuffle=True)

# Training loop
for epoch in range(num_epochs):
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")


## 6. Model Evaluation

Evaluating the model’s performance on the test set helps us understand how well it generalizes to unseen data. We use the following steps to assess the model:

1. **Set the Model to Evaluation Mode**
   - `model.eval()` changes the model’s behavior for inference, disabling dropout and other training-specific settings.
   
2. **Generate Predictions**
   - We use `torch.no_grad()` to prevent gradient calculations, which reduces memory usage and speeds up computations.
   - The model generates predictions for each sequence in the test set.

3. **Calculate Mean Squared Error (MSE)**
   - MSE is a common evaluation metric for regression tasks, providing a measure of the average squared difference between predicted and actual values.
   - Lower MSE values indicate that the model’s predictions are closer to the actual values.
   

In [ ]:
# Evaluation
model.eval()
with torch.no_grad():
    predictions = model(torch.tensor(X_test, dtype=torch.float32)).squeeze()
    mse = criterion(predictions, torch.tensor(y_test, dtype=torch.float32)).item()
print(f"Test Mean Squared Error: {mse:.4f}")


## 7. Model Interpretation with Integrated Gradients from the Captum library

To understand how each feature influences the model’s predictions, we use **Integrated Gradients** from the `captum` library. Integrated Gradients is an interpretability technique that calculates the contribution of each input feature to the model’s prediction, helping us identify which features and time steps are most important.


In [ ]:
!pip install captum

In [ ]:
from captum.attr import IntegratedGradients

# Initialize Integrated Gradients
ig = IntegratedGradients(model)

# Choose a test instance for interpretation
test_instance = torch.tensor(X_test[0], dtype=torch.float32).unsqueeze(0).requires_grad_()

# Calculate attributions
attributions, _ = ig.attribute(test_instance, target=0, return_convergence_delta=True)

# Plot feature attributions
attributions = attributions.squeeze().detach().numpy()
features = ['precipitation', 'temp_max', 'temp_min', 'wind', 'weather_encoded']

plt.figure(figsize=(10, 6))
for i, feature in enumerate(features):
    plt.plot(attributions[:, i], label=feature)
plt.xlabel('Time Steps')
plt.ylabel('Attribution')
plt.title('Feature Attributions for Time Series Prediction')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Reshape attributions for easier manipulation (if necessary)
attributions = attributions.squeeze()

# Cumulative Attribution Plot
cumulative_attributions = np.sum(attributions, axis=0)
plt.figure(figsize=(8, 5))
plt.bar(features, cumulative_attributions)
plt.title("Cumulative Feature Attributions")
plt.ylabel("Cumulative Attribution")
plt.xlabel("Features")
plt.show()

In [ ]:
# Heatmap of Attributions
plt.figure(figsize=(10, 6))
sns.heatmap(attributions.T, cmap="YlGnBu", annot=False, xticklabels=5, yticklabels=features)
plt.title("Heatmap of Feature Attributions Over Time Steps")
plt.xlabel("Time Steps")
plt.ylabel("Features")
plt.show()

In [ ]:
# Bar Plot for Last Time Step
plt.figure(figsize=(8, 5))
plt.bar(features, attributions[-1])
plt.title("Feature Attributions at Last Time Step")
plt.ylabel("Attribution")
plt.xlabel("Features")
plt.show()

## XAI (Explainable AI) Results Interpretation

The following plots provide insights into the contributions of each feature in predicting the target variable over a sequence of 30 time steps using Integrated Gradients.

### 1. Feature Attributions for Time Series Prediction

- **Interpretation**:
  - This line plot tracks each feature's attribution over the sequence of 30 time steps.
  - **`temp_max`** shows a significant upward trend in later time steps, meaning its influence increases as the sequence progresses.
  - **`temp_min`** and **`weather_encoded`** also increase in influence over time, but to a lesser extent.
  - **`precipitation`** and **`wind`** maintain consistently low attributions, implying minimal influence at all points in the sequence.

### 2. Cumulative Feature Attributions

- **Interpretation**:
  - This bar chart summarizes the overall impact of each feature across all time steps by showing the cumulative attribution for each.
  - **`temp_max`** has the highest cumulative attribution, followed by **`temp_min`** and **`weather_encoded`**. This indicates that these features are the most influential in the model’s prediction.
  - **`precipitation`** and **`wind`** have low cumulative attributions, suggesting they contribute less to the prediction.

### 3. Heatmap of Feature Attributions Over Time Steps

- **Interpretation**:
  - This heatmap visualizes the attributions for each feature across all 30 time steps.
  - The intensity of the color represents the magnitude of the attribution. Darker colors indicate higher attributions.
  - **`temp_max`** and **`temp_min`** show stronger attributions in the later time steps, emphasizing that recent values for these features significantly impact the model's prediction.
  - **`weather_encoded`** also has moderate attribution in the later steps, while **`precipitation`** and **`wind`** remain low throughout.

### 4. Feature Attributions at Last Time Step

- **Interpretation**:
  - This bar chart shows the attribution values for each feature at the final time step (time step 30).
  - **`temp_max`** and **`temp_min`** have the highest attributions, indicating that the model relies heavily on the most recent values of these features to make predictions.
  - **`weather_encoded`** also has a noticeable contribution, whereas **`precipitation`** and **`wind`** have minimal impact at this step.



### Summary

- **Key Influential Features**: `temp_max` and `temp_min` are the most influential features, particularly in the later time steps, which suggests that the model relies on recent temperature data for predictions.
- **Consistently Low-Influence Features**: `precipitation` and `wind` have minimal impact on the model's predictions throughout the sequence.
- **Moderate Influence**: `weather_encoded` contributes moderately to the prediction, with its influence increasing toward the end of the sequence.

These interpretations provide valuable insights into how each feature contributes to the model’s decisions, highlighting the importance of temperature-related features in the model's predictions.
